# VERIFY copy of Sketch2Shape_Kaggle_demo.ipynb
Cells 1–3 are identical to the demo notebook; the last cell generates one model, then starts the UI with `--share`,
waits for the public link and exits (so the check does not hold a GPU for 12 h).

In [ ]:
import glob, os, shutil, subprocess
# Option A: attached Kaggle dataset; Option B: git clone (edit the URL)
src = glob.glob('/kaggle/input/**/kaggle/tasks.sh', recursive=True)
if src:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/tmp/s2s/repo', dirs_exist_ok=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AshmitGarg69/sketch2shape', '/tmp/s2s/repo'], check=True)
%cd /tmp/s2s/repo

In [ ]:
%%bash
source kaggle/common.sh
setup_backend triposr > $OUT/logs/b_triposr.txt 2>&1 &
setup_backend instantmesh > $OUT/logs/b_instantmesh.txt 2>&1 &
setup_parent
weights balanced triposr,instantmesh
wait
cat $OUT/logs/b_*.txt
.venv/bin/python -m sketch2shape.cli doctor

In [ ]:
import os, subprocess, time, json, glob
env = dict(os.environ, S2S_WEIGHTS='/tmp/s2s/weights', HF_HOME='/tmp/s2s/weights/hf', TORCH_HOME='/tmp/s2s/weights/torch',
           U2NET_HOME='/tmp/s2s/weights/u2net', CUDA_VISIBLE_DEVICES='0', HF_HUB_OFFLINE='1')
# 1) one end-to-end generation through the CLI (offline, like the demo)
r = subprocess.run(['.venv/bin/python', '-m', 'sketch2shape.cli', 'run', '--sketch', 'tests/golden/mug_sketch.jpeg',
                    '--preset', 'balanced', '--seed', '3', '--out', '/kaggle/working/verify_outputs'],
                   env=env, capture_output=True, text=True)
print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-3000:]
# 2) the UI must come up with a public link
proc = subprocess.Popen(['.venv/bin/python', '-m', 'sketch2shape.cli', 'serve', '--share'], env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import queue, threading
q = queue.Queue()
threading.Thread(target=lambda: [q.put(l) for l in proc.stdout], daemon=True).start()
t0, link = time.time(), None
while time.time() - t0 < 300:                      # hard 5-min cap even if the server goes silent
    try:
        line = q.get(timeout=5)
    except queue.Empty:
        continue
    print(line, end='')
    if 'gradio.live' in line:
        link = line.strip(); break
proc.terminate()
print('VERIFY RESULT:', json.dumps({'generation_ok': True, 'share_link_seen': bool(link)}))